# Workspace metadata sync validation and upgrade

**Default mode: validation only.** This notebook discovers all lakehouses and SQL
analytics endpoints in the configured Fabric workspace, including nested folders,
checks the metadata sync architecture, and assesses each legacy endpoint.
It creates/appends `lakehouse.dbo.md_sync_audit`; it does not change metadata sync
unless an upgrade switch below is explicitly enabled.

The audit lakehouse is the lakehouse linked in the request. All responses are
retained in Delta, including raw response bodies, safe response headers,
asynchronous operation IDs and states, endpoint snapshots, table blockers,
table migration results, and the final decision for each endpoint.
Every run has its own `run_id`; historical rows are never overwritten.

| UPGRADE_IF_ALL_ELIGIBLE | FORCE_UPGRADE | Behavior per endpoint |
| --- | --- | --- |
| False | False | Assess and log only. |
| True | False | Upgrade only when every evaluated table is eligible and no blockers exist. |
| False or True | True | After a successful assessment, attempt upgrade even when blockers exist. |

**Important:** Force is a notebook-side override, not a REST API parameter.
It cannot make unsupported tables compatible. If some tables upgrade and others
fail, the endpoint can report **V2 while failed tables remain V1 and no longer
sync automatically**. The notebook detects these mixed/partial states and can
assess/retry remaining tables on later runs. Healthy V2 endpoints are skipped.

Empty endpoints are logged and skipped. Active migrations, unknown API contracts,
failed assessments, and changed endpoint versions are never bypassed by force.
Preflight is a point-in-time assessment, not a guarantee against subsequent
table changes. Perform upgrades in a maintenance window with concurrent changes
controlled; this notebook does not rewrite Delta tables or remediate blockers.


## Configuration

Edit these parameters, then run all cells in order. Keep both upgrade switches False for assessment only.


In [ ]:
WORKSPACE_ID = "b5f7d625-e995-42e5-8d93-df8612e11b02"
FABRIC_API_BASE_URL = "https://api.fabric.microsoft.com/v1"

UPGRADE_IF_ALL_ELIGIBLE = True
FORCE_UPGRADE = False

AUDIT_LAKEHOUSE_ID = "fc831e00-927c-4ba7-8c19-7f79ad2efa4c"
AUDIT_LAKEHOUSE_WORKSPACE_ID = "b5f7d625-e995-42e5-8d93-df8612e11b02"
AUDIT_LAKEHOUSE_NAME = "lakehouse"
AUDIT_SCHEMA_NAME = "dbo"
AUDIT_TABLE_NAME = "md_sync_audit"

HTTP_TIMEOUT_SECONDS = 120
OPERATION_TIMEOUT_SECONDS = 3600
POLL_INTERVAL_SECONDS = 10
MAX_RETRIES = 5
FAIL_ON_ERRORS = True


## REST client and upgrade policy

The following self-contained helpers implement discovery, contract validation, upgrade gating, asynchronous polling, and endpoint processing.


In [ ]:
"""Self-contained, fail-closed Fabric metadata-sync audit/upgrade engine.

Public API: Config, HttpReply, FabricError, TransportError,
Client(config, token_provider, transport, audit, clock=time.monotonic,
       sleep=time.sleep), Client.call(method, path, stage,
       endpoint_context=None, deadline=None), Client.operation(method, path,
       stage, endpoint_context=None), Client.list_all(collection),
run_workspace(client, audit). Client.last_operation_id identifies the most
recent operation; operation() returns the unmodified final result object.

The audit sink implements emit(stage, **fields) and flush(). Sink exceptions
must NOT inherit FabricError. No sink exceptions or programming errors are
caught. HTTP requests, raw responses, decisions and accepted operation IDs are
flushed immediately; table events are flushed no later than endpoint completion.
The transport must disable redirects and raise TransportError without secrets.
Only the public https://api.fabric.microsoft.com/v1 API base is accepted.
LRO polling always uses canonical API operation URLs. A validated
x-ms-operation-id (or an already resolved operation ID) is authoritative;
Location is validated only when it is the sole routing source. Foreign
Location headers remain audited but never receive credentials.
Listing continuation tokens take precedence over continuation URIs and build
canonical recursive collection URLs. URI-only continuations require same-origin
HTTPS validation.

Summary fields (one per distinct SQL endpoint):
endpoint_id, endpoint_name, lakehouse_id, lakehouse_name, version_before,
version_after, migration_status, evaluated_table_count, eligible_table_count,
blocker_count, all_tables_eligible, upgrade_attempted, outcome, message,
operation_id.

Summary outcome values: ALREADY_V2, MIGRATION_IN_PROGRESS, NO_TABLES,
VALIDATION_READY, VALIDATION_BLOCKED, UPGRADE_SKIPPED_BLOCKERS, UPGRADED,
UPGRADE_PARTIAL, UPGRADE_FAILED, ERROR, STATE_CHANGED_RERUN, ENDPOINT_NOT_READY.
ENDPOINT_MISSING is a lakehouse-only skip event, not an endpoint summary.

Migration attempts accept both the documented long field names and the
sourceVersion/targetVersion/status/completedAt fields returned by MSIT.
Known migration outcomes are normalized case-insensitively; conflicting
aliases and unknown values remain contract errors. Raw responses are unchanged.

Stable audit projections, in addition to stage and workspace_id:
endpoint_id, endpoint_name, lakehouse_id, lakehouse_name, event_state,
http_method, request_url, http_status, request_id, operation_id,
operation_state, sync_version, migration_direction, migration_status,
table_schema, table_name, table_sync_version, last_successful_sync_utc,
evaluated_table_count, eligible_table_count, blocker_count,
all_tables_eligible, upgrade_attempted, error_code, message, response_body,
response_headers_json. Exact additional event fields:
action, attempt, retry, reason, wait_seconds, remaining_seconds, status_phase,
collection, page_number, item_count, new_item_count, continuation_uri,
continuation_token, provisioning_status, endpoint_ready, last_migration_attempt,
corrective_action, parsed_result, operation_details, table_failure_count,
upgrade_planned, outcome, version_before, version_after,
upgrade_if_all_eligible, force_upgrade, api_base_url, all_endpoints_processed,
lakehouse_count, listed_endpoint_count, endpoint_count,
processed_endpoint_count, outcome_counts.

Principal stages: RUN_START, RUN_COMPLETE, RUN_FAILED, DISCOVERY_ERROR,
DISCOVERY_PAGE, LAKEHOUSE_DISCOVERED, LAKEHOUSE_SKIPPED, ENDPOINT_ASSOCIATION,
ENDPOINT_DISCOVERED, ENDPOINT_START, HTTP_REQUEST, HTTP_RESPONSE,
HTTP_TRANSPORT_ERROR, HTTP_RETRY_DECISION, WAIT, TIMEOUT,
OPERATION_ACCEPTED, OPERATION_STATE, OPERATION_RESULT_PENDING,
OPERATION_RESULT, OPERATION_ERROR, MIGRATION_STATUS, MIGRATION_ATTEMPT,
TABLE_MIGRATION_FAILURE, PREFLIGHT_RESULT, TABLE_BLOCKER, UPGRADE_DECISION,
TABLE_UPGRADE_STATUS, UPGRADE_RESULT, UPGRADE_ERROR, POST_STATUS_ERROR,
ENDPOINT_DECISION, ENDPOINT_ERROR, ENDPOINT_SUMMARY.

Preflight is a point-in-time, non-mutating check, not a guarantee. Neither mode
changes Delta tables or takes corrective actions. Force only bypasses table
eligibility blockers after a successful, well-formed, nonempty preflight; there
is no server-side force argument. Failed/partial upgrades never mean success.
"""

import json
import re
import time
import uuid
from dataclasses import dataclass
from datetime import datetime
from urllib.parse import urlencode, urljoin, urlsplit


class FabricError(RuntimeError):
    """An anticipated API, transport, response-contract or LRO failure."""

    def __init__(self, message, *, error_code="FABRIC_ERROR", operation_id=None):
        super().__init__(message)
        self.error_code = error_code
        self.operation_id = operation_id


class TransportError(FabricError):
    """Transport failures; callers must not include credential details."""


def _uuid(value, label):
    if not isinstance(value, str):
        raise FabricError(label + " must be a UUID", error_code="INVALID_ID")
    try:
        return str(uuid.UUID(value))
    except ValueError:
        raise FabricError(label + " must be a UUID", error_code="INVALID_ID") from None


@dataclass(frozen=True)
class Config:
    workspace_id: str
    api_base_url: str = "https://api.fabric.microsoft.com/v1"
    upgrade_if_all_eligible: bool = False
    force_upgrade: bool = False
    http_timeout_seconds: int = 120
    operation_timeout_seconds: int = 3600
    poll_interval_seconds: int = 10
    max_retries: int = 5

    def __post_init__(self):
        try:
            workspace_id = _uuid(self.workspace_id, "workspace_id")
        except FabricError as exc:
            raise ValueError(str(exc)) from None
        object.__setattr__(self, "workspace_id", workspace_id)
        for name in ("upgrade_if_all_eligible", "force_upgrade"):
            if type(getattr(self, name)) is not bool:
                raise ValueError(name + " must be an actual bool")
        for name in ("http_timeout_seconds", "operation_timeout_seconds",
                     "poll_interval_seconds", "max_retries"):
            value = getattr(self, name)
            minimum = 0 if name == "max_retries" else 1
            if type(value) is not int or value < minimum:
                raise ValueError(name + " must be an integer >= " + str(minimum))
        if not isinstance(self.api_base_url, str):
            raise ValueError("api_base_url must be the public HTTPS Fabric v1 base")
        try:
            parsed = urlsplit(self.api_base_url)
            valid = (
                parsed.scheme == "https"
                and parsed.hostname == "api.fabric.microsoft.com"
                and parsed.port in (None, 443)
                and parsed.username is None and parsed.password is None
                and parsed.path in ("/v1", "/v1/")
                and not parsed.query and not parsed.fragment
                and not any(ord(c) <= 32 or ord(c) == 127 or c == "\\"
                            for c in self.api_base_url)
            )
        except ValueError:
            valid = False
        if not valid:
            raise ValueError("api_base_url must be the public HTTPS Fabric v1 base")
        object.__setattr__(self, "api_base_url", "https://api.fabric.microsoft.com/v1")


@dataclass(frozen=True)
class HttpReply:
    status: int
    headers: dict[str, str]
    text: str


_SAFE_HEADERS = {
    name.lower(): name for name in (
        "Content-Type", "Location", "Retry-After", "x-ms-operation-id",
        "requestId", "x-ms-request-id", "ActivityId",
    )
}
_CONTEXT_FIELDS = (
    "endpoint_id", "endpoint_name", "lakehouse_id", "lakehouse_name", "operation_id",
)
_VERSIONS = ("V1", "V2")
_MIGRATION_RESULTS = ("Succeeded", "PartiallySucceeded", "Failed")
_CANCEL_STATES = ("Cancelled", "Canceled", "Cancelling", "Canceling")
_SUMMARY_FIELDS = (
    "endpoint_id", "endpoint_name", "lakehouse_id", "lakehouse_name",
    "version_before", "version_after", "migration_status",
    "evaluated_table_count", "eligible_table_count", "blocker_count",
    "all_tables_eligible", "upgrade_attempted", "outcome", "message", "operation_id",
)


def _write(audit, stage, context=None, *, flush=False, **fields):
    event = dict(context or {})
    event.update(fields)
    audit.emit(stage, **event)
    if flush:
        audit.flush()


def _contract(message):
    raise FabricError(message, error_code="INVALID_RESPONSE")


def _object(reply, label):
    def invalid_constant(_):
        raise ValueError("Non-JSON number")

    def unique_object(pairs):
        body = {}
        for key, value in pairs:
            if key in body:
                raise ValueError("Duplicate JSON field")
            body[key] = value
        return body

    try:
        body = json.loads(reply.text, parse_constant=invalid_constant,
                          object_pairs_hook=unique_object)
    except ValueError:
        _contract(label + " is not valid JSON")
    if not isinstance(body, dict):
        _contract(label + " must be a JSON object")
    return body


def _header(reply, name):
    for key, value in reply.headers.items():
        if key.lower() == name.lower():
            return value
    return None


def _safe_headers(reply):
    return {_SAFE_HEADERS[key.lower()]: value
            for key, value in reply.headers.items() if key.lower() in _SAFE_HEADERS}


def _string_field(body, name, label, *, allow_empty=False):
    value = body.get(name)
    if not isinstance(value, str) or (not allow_empty and not value.strip()):
        _contract(label + "." + name + " must be a string")
    return value


def _timestamp_field(body, name, label):
    value = _string_field(body, name, label)
    try:
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        _contract(label + "." + name + " must be an ISO timestamp")
    if parsed.tzinfo is None:
        _contract(label + "." + name + " must include a time zone")
    return value


def _error_shape(error, label):
    if not isinstance(error, dict):
        _contract(label + " must be an error object")
    _string_field(error, "errorCode", label)
    _string_field(error, "message", label, allow_empty=True)


def _table_identity(table, label):
    if not isinstance(table, dict):
        _contract(label + " must be an object")
    _string_field(table, "schema", label)
    _string_field(table, "tableName", label)


def _migration_outcome(value, label):
    if isinstance(value, str):
        for outcome in _MIGRATION_RESULTS:
            if value.casefold() == outcome.casefold():
                return outcome
    _contract(label + " has an unknown or missing migration outcome")


def _attempt_field(attempt, canonical, alias, validate):
    values = []
    for name in (canonical, alias):
        if name in attempt:
            values.append(validate(attempt[name], "lastMigrationAttempt." + name))
    if not values:
        _contract("lastMigrationAttempt." + canonical + " is missing")
    if any(value != values[0] for value in values[1:]):
        _contract("lastMigrationAttempt has conflicting " + canonical + "/" + alias)
    return values[0]


def _attempt_version(value, label):
    if value not in _VERSIONS:
        _contract(label + " is invalid")
    return value


def _attempt_timestamp(value, label):
    return _timestamp_field({"timestamp": value}, "timestamp", label)


def _normalize_attempt(attempt):
    if not isinstance(attempt, dict):
        _contract("lastMigrationAttempt must be an object")
    normalized = dict(attempt)
    for canonical, alias, validate in (
        ("sourceSyncArchitectureVersion", "sourceVersion", _attempt_version),
        ("targetSyncArchitectureVersion", "targetVersion", _attempt_version),
        ("itemMigrationStatus", "status", _migration_outcome),
        ("completedDateTime", "completedAt", _attempt_timestamp),
    ):
        normalized[canonical] = _attempt_field(attempt, canonical, alias, validate)
    return normalized


def _status_shape(body):
    if body.get("syncArchitectureVersion") not in _VERSIONS:
        _contract("Migration status has an unknown or missing syncArchitectureVersion")
    if ("itemMigrationDirection" in body
            and body["itemMigrationDirection"] not in ("MigratingToV2", "MigratingToV1")):
        _contract("Migration status has an invalid itemMigrationDirection")
    failures = body.get("tableFailures")
    if not isinstance(failures, list):
        _contract("Migration status.tableFailures must be a list")
    for failure in failures:
        _table_identity(failure, "tableFailures entry")
        _error_shape(failure.get("error"), "tableFailures entry.error")
    normalized = dict(body)
    if "lastMigrationAttempt" in body:
        normalized["lastMigrationAttempt"] = _normalize_attempt(body["lastMigrationAttempt"])
    return normalized


def _partial_attempt(body):
    attempt = body.get("lastMigrationAttempt", {})
    return (
        attempt.get("sourceSyncArchitectureVersion") == "V1"
        and attempt.get("targetSyncArchitectureVersion") == "V2"
        and attempt.get("itemMigrationStatus") in ("PartiallySucceeded", "Failed")
    )


def _preflight_shape(body):
    evaluated = body.get("evaluatedTableCount")
    eligible = body.get("eligibleTableCount")
    if (type(evaluated) is not int or type(eligible) is not int
            or evaluated < 0 or eligible < 0 or eligible > evaluated):
        _contract("Preflight requires integer counts with 0 <= eligible <= evaluated")
    blockers = body.get("blockers")
    if not isinstance(blockers, list):
        _contract("Preflight.blockers must be a list")
    for blocker in blockers:
        _table_identity(blocker, "blockers entry")
        _string_field(blocker, "correctiveAction", "blockers entry", allow_empty=True)
    return body


def _upgrade_shape(body):
    if body.get("syncArchitectureVersion") not in _VERSIONS:
        _contract("Upgrade result has an unknown or missing syncArchitectureVersion")
    normalized = dict(body)
    normalized["itemMigrationStatus"] = _migration_outcome(
        body.get("itemMigrationStatus"), "Upgrade result.itemMigrationStatus")
    tables = body.get("tableStatuses")
    if not isinstance(tables, list) or not tables:
        _contract("Upgrade result must report a nonempty tableStatuses list")
    for table in tables:
        _table_identity(table, "tableStatuses entry")
        if table.get("syncArchitectureVersion") not in _VERSIONS:
            _contract("tableStatuses entry.syncArchitectureVersion is invalid")
        if table.get("error") is not None:
            _error_shape(table["error"], "tableStatuses entry.error")
        if "lastSuccessfulSyncDateTime" in table:
            _timestamp_field(table, "lastSuccessfulSyncDateTime", "tableStatuses entry")
    return normalized


class Client:
    def __init__(self, config, token_provider, transport, audit,
                 clock=time.monotonic, sleep=time.sleep):
        if not isinstance(config, Config):
            raise TypeError("config must be a Config")
        if not all(callable(value) for value in (token_provider, transport, clock, sleep)):
            raise TypeError("token_provider, transport, clock and sleep must be callable")
        if not callable(getattr(audit, "emit", None)) or not callable(getattr(audit, "flush", None)):
            raise TypeError("audit must implement emit and flush")
        self.config = config
        self.token_provider = token_provider
        self.transport = transport
        self.audit = audit
        self.clock = clock
        self.sleep = sleep
        self.last_operation_id = None

    def _event(self, stage, context=None, *, flush=False, **fields):
        context = {key: value for key, value in (context or {}).items()
                   if key in _CONTEXT_FIELDS}
        context["workspace_id"] = self.config.workspace_id
        _write(self.audit, stage, context, flush=flush, **fields)

    def _validate_url(self, url):
        if (not isinstance(url, str) or url.startswith("//")
                or any(ord(c) <= 32 or ord(c) == 127 or c == "\\" for c in url)):
            raise FabricError("Unsafe API URL", error_code="UNSAFE_URL")
        try:
            parsed = urlsplit(url)
            valid = (
                parsed.scheme == "https"
                and parsed.hostname == "api.fabric.microsoft.com"
                and parsed.port in (None, 443)
                and parsed.username is None and parsed.password is None
                and not parsed.fragment
                and parsed.path.startswith("/v1/")
            )
        except ValueError:
            valid = False
        if not valid:
            raise FabricError("API URL must stay on the configured HTTPS origin",
                              error_code="UNSAFE_URL")
        return url

    def _url(self, path):
        if not isinstance(path, str) or not path or path.startswith("//"):
            raise FabricError("Invalid API path", error_code="UNSAFE_URL")
        try:
            parsed = urlsplit(path)
        except ValueError:
            raise FabricError("Malformed API URL", error_code="UNSAFE_URL") from None
        if parsed.scheme or parsed.netloc:
            url = path
        elif path.startswith("/v1/"):
            url = "https://api.fabric.microsoft.com" + path
        else:
            url = self.config.api_base_url + "/" + path.lstrip("/")
        return self._validate_url(url)

    def _joined_url(self, base, reference):
        try:
            url = urljoin(base, reference)
        except ValueError:
            raise FabricError("Malformed continuation/operation URL",
                              error_code="UNSAFE_URL") from None
        return self._validate_url(url)

    def _remaining(self, deadline, stage, context):
        remaining = deadline - self.clock()
        if remaining <= 0:
            self._event("TIMEOUT", context, flush=True, action=stage,
                        event_state="TIMEOUT", error_code="TIMEOUT",
                        message="The request/operation deadline expired")
            raise FabricError("The request/operation deadline expired", error_code="TIMEOUT")
        return remaining

    def _retry_after(self, reply):
        raw = _header(reply, "Retry-After")
        if raw is None:
            return self.config.poll_interval_seconds
        if not re.fullmatch(r"[0-9]+", raw.strip()):
            _contract("Retry-After must contain nonnegative integer seconds")
        try:
            return int(raw.strip())
        except ValueError:
            _contract("Retry-After is not a usable integer")

    def _wait(self, seconds, deadline, stage, context, reason):
        remaining = self._remaining(deadline, stage, context)
        if seconds >= remaining:
            self._event("TIMEOUT", context, flush=True, action=stage,
                        event_state="TIMEOUT", error_code="TIMEOUT",
                        wait_seconds=seconds, remaining_seconds=remaining,
                        message="Required wait would exhaust the operation deadline")
            raise FabricError("Required wait would exhaust the operation deadline",
                              error_code="TIMEOUT")
        self._event("WAIT", context, flush=True, action=stage,
                    wait_seconds=seconds, reason=reason, event_state="WAITING")
        if seconds:
            self.sleep(seconds)
        self._remaining(deadline, stage, context)

    def call(self, method, path, stage, endpoint_context=None, deadline=None):
        """Audit each attempt before/after I/O; retry mutations only on 429/one 401."""
        if not isinstance(method, str) or method.upper() not in (
                "GET", "POST", "PUT", "PATCH", "DELETE", "HEAD", "OPTIONS"):
            raise ValueError("Unsupported HTTP method")
        method = method.upper()
        url = self._url(path)
        deadline = self.clock() + self.config.operation_timeout_seconds if deadline is None else deadline
        retries, credential_retry, attempt = 0, False, 0
        while True:
            remaining = self._remaining(deadline, stage, endpoint_context)
            attempt += 1
            fields = dict(action=stage, http_method=method, request_url=url, attempt=attempt)
            self._event("HTTP_REQUEST", endpoint_context, flush=True,
                        event_state="SENDING", **fields)
            token = self.token_provider()
            if not isinstance(token, str) or not token.strip():
                raise FabricError("Token provider returned no credential", error_code="TOKEN_UNAVAILABLE")
            headers = {"Authorization": "Bearer " + token, "Accept": "application/json"}
            del token
            try:
                remaining = self._remaining(deadline, stage, endpoint_context)
                reply = self.transport(method, url, headers,
                                       min(self.config.http_timeout_seconds, remaining))
            except TransportError:
                self._event("HTTP_TRANSPORT_ERROR", endpoint_context, flush=True,
                            event_state="AMBIGUOUS", error_code="TRANSPORT_ERROR",
                            message="Transport failed; the server outcome may be unknown", **fields)
                retry = method == "GET" and retries < self.config.max_retries
                self._event("HTTP_RETRY_DECISION", endpoint_context, flush=True, retry=retry,
                            reason="GET_TRANSPORT_RETRY" if retry else "NO_AMBIGUOUS_REPLAY",
                            **fields)
                if not retry:
                    raise TransportError("Transport failed; the server outcome may be unknown",
                                         error_code="TRANSPORT_ERROR") from None
                retries += 1
                self._wait(self.config.poll_interval_seconds, deadline, stage,
                           endpoint_context, "GET_TRANSPORT_RETRY")
                continue
            finally:
                del headers
            if (not isinstance(reply, HttpReply) or type(reply.status) is not int
                    or not 100 <= reply.status <= 599 or not isinstance(reply.text, str)
                    or not isinstance(reply.headers, dict)
                    or not all(isinstance(k, str) and isinstance(v, str)
                               for k, v in reply.headers.items())):
                raise FabricError("Transport returned an invalid HttpReply",
                                  error_code="TRANSPORT_CONTRACT")
            safe = _safe_headers(reply)
            self._event(
                "HTTP_RESPONSE", endpoint_context, flush=True, **fields,
                event_state="ACCEPTED" if 200 <= reply.status < 300 else "REJECTED",
                http_status=reply.status, response_body=reply.text,
                response_headers_json=json.dumps(safe, sort_keys=True),
                request_id=(_header(reply, "requestId") or _header(reply, "x-ms-request-id")
                            or _header(reply, "ActivityId")),
                operation_id=(_header(reply, "x-ms-operation-id")
                              or (endpoint_context or {}).get("operation_id")),
            )
            self._remaining(deadline, stage, endpoint_context)
            if 200 <= reply.status < 300:
                return reply
            retry, reason, delay = False, "NOT_RETRYABLE", 0
            if reply.status == 401 and not credential_retry:
                credential_retry, retry, reason = True, True, "CREDENTIAL_REFRESH"
            elif reply.status == 429 and retries < self.config.max_retries:
                retry, reason = True, "THROTTLED"
                retries += 1
            elif 500 <= reply.status < 600 and method == "GET" and retries < self.config.max_retries:
                retry, reason = True, "GET_SERVER_RETRY"
                retries += 1
            elif reply.status == 401 or reply.status == 429 or 500 <= reply.status < 600:
                reason = "RETRY_LIMIT_OR_NON_IDEMPOTENT"
            if retry and reason in ("THROTTLED", "GET_SERVER_RETRY"):
                try:
                    delay = self._retry_after(reply)
                except FabricError as exc:
                    self._event("HTTP_RETRY_DECISION", endpoint_context, flush=True,
                                retry=False, reason="INVALID_RETRY_AFTER",
                                http_status=reply.status, error_code=exc.error_code, **fields)
                    raise
            self._event("HTTP_RETRY_DECISION", endpoint_context, flush=True,
                        retry=retry, reason=reason, wait_seconds=delay,
                        http_status=reply.status, **fields)
            if not retry:
                raise FabricError("Fabric returned HTTP " + str(reply.status),
                                  error_code="HTTP_" + str(reply.status))
            self._wait(delay, deadline, stage, endpoint_context, reason)

    def _operation_id(self, reply, existing=None):
        supplied = _header(reply, "x-ms-operation-id")
        if supplied is not None:
            operation_id = _uuid(supplied, "Operation ID")
            if existing is not None and operation_id != existing:
                _contract("An operation response changed the operation ID")
            return operation_id
        if existing is not None:
            return existing
        location = _header(reply, "Location")
        if location is None:
            _contract("HTTP 202 is missing a usable operation ID or Location")
        location_url = self._joined_url(self.config.api_base_url + "/", location)
        parsed = urlsplit(location_url)
        match = re.fullmatch(r"/v1/operations/([^/]+)(?:/result)?/?", parsed.path)
        if match is None or parsed.query:
            _contract("Location must identify a Fabric operation")
        return _uuid(match.group(1), "Location operation ID")

    def operation(self, method, path, stage, endpoint_context=None):
        """Resolve an LRO without replaying its initial mutation on ambiguous errors."""
        self.last_operation_id = None
        deadline = self.clock() + self.config.operation_timeout_seconds
        context = dict(endpoint_context or {})
        try:
            initial = self.call(method, path, stage, context, deadline)
            if initial.status in (200, 201):
                result = _object(initial, stage + " result")
                self._event("OPERATION_RESULT", context, flush=True, action=stage,
                            event_state="COMPLETED", parsed_result=result)
                return result
            if initial.status != 202:
                _contract("Operation requires HTTP 200, 201 or 202")
            operation_id = self._operation_id(initial)
            self.last_operation_id = operation_id
            context["operation_id"] = operation_id
            self._event("OPERATION_ACCEPTED", context, flush=True, action=stage,
                        event_state="ACCEPTED", http_status=202, response_body=initial.text,
                        response_headers_json=json.dumps(_safe_headers(initial), sort_keys=True))
            self._wait(self._retry_after(initial), deadline, stage, context, "INITIAL_POLL")
            while True:
                state_reply = self.call("GET", "/operations/" + operation_id,
                                        stage + "_POLL", context, deadline)
                if state_reply.status != 200:
                    _contract("Operation state requires HTTP 200")
                state_body = _object(state_reply, "Operation state")
                state = state_body.get("status")
                self._event("OPERATION_STATE", context, flush=True, action=stage,
                            operation_state=state if isinstance(state, str) else None,
                            operation_details=state_body,
                            event_state="STATE", response_body=state_reply.text)
                if state == "Failed" or state in _CANCEL_STATES:
                    code = "OPERATION_FAILED" if state == "Failed" else "OPERATION_CANCELLED"
                    raise FabricError("Fabric operation " + state, error_code=code,
                                      operation_id=operation_id)
                if state in ("NotStarted", "Running"):
                    self._wait(self._retry_after(state_reply), deadline, stage, context, "STATE_POLL")
                    continue
                if state != "Succeeded":
                    _contract("Operation state is missing or unknown")
                result_reply = self.call("GET", "/operations/" + operation_id + "/result",
                                         stage + "_RESULT", context, deadline)
                if result_reply.status in (200, 201):
                    result = _object(result_reply, stage + " result")
                    self._event("OPERATION_RESULT", context, flush=True, action=stage,
                                event_state="COMPLETED", operation_state="Succeeded",
                                parsed_result=result)
                    return result
                if result_reply.status != 202:
                    _contract("Operation result requires HTTP 200, 201 or 202")
                self._operation_id(result_reply, operation_id)
                self._event("OPERATION_RESULT_PENDING", context, flush=True, action=stage,
                            event_state="RESULT_PENDING", operation_state="Succeeded",
                            response_body=result_reply.text)
                self._wait(self._retry_after(result_reply), deadline, stage, context, "RESULT_POLL")
        except FabricError as exc:
            if exc.operation_id is None:
                exc.operation_id = self.last_operation_id
            self._event("OPERATION_ERROR", context, flush=True, action=stage,
                        event_state="FAILED", error_code=exc.error_code, message=str(exc))
            raise

    def list_all(self, collection):
        """Recursively list lakehouses/sqlEndpoints with pagination and UUID deduplication."""
        if collection not in ("lakehouses", "sqlEndpoints"):
            raise ValueError("collection must be lakehouses or sqlEndpoints")
        root = "/workspaces/" + self.config.workspace_id + "/" + collection
        url = self._url(root + "?recursive=true")
        deadline = self.clock() + self.config.operation_timeout_seconds
        seen_urls, seen_tokens, seen_pages, items = set(), set(), set(), {}
        page_number = 0
        while True:
            if url in seen_urls:
                raise FabricError("Pagination repeated a continuation URL", error_code="PAGINATION_CYCLE")
            seen_urls.add(url)
            reply = self.call("GET", url, "LIST_" + collection, deadline=deadline)
            if reply.status != 200:
                _contract("Collection listing requires HTTP 200")
            body = _object(reply, "Collection listing")
            values = body.get("value")
            if not isinstance(values, list):
                _contract("Collection listing.value must be a list")
            continuation_uri, continuation_token = body.get("continuationUri"), body.get("continuationToken")
            for name, value in (("continuationUri", continuation_uri),
                                ("continuationToken", continuation_token)):
                if value is not None and not isinstance(value, str):
                    _contract("Collection listing." + name + " must be a string")
            page_items = []
            for item in values:
                if not isinstance(item, dict):
                    _contract("Collection entries must be objects")
                item_id = _uuid(item.get("id"), "Collection item ID")
                if "workspaceId" in item and _uuid(item["workspaceId"], "Item workspace ID") != self.config.workspace_id:
                    _contract("Collection item belongs to a different workspace")
                expected_type = "Lakehouse" if collection == "lakehouses" else "SQLEndpoint"
                if "type" in item and item["type"] != expected_type:
                    _contract("Collection contains an unexpected item type")
                if "displayName" in item:
                    _string_field(item, "displayName", "Collection item")
                page_items.append(dict(item, id=item_id))
            signature = tuple(sorted({item["id"] for item in page_items}))
            if signature in seen_pages and (continuation_uri or continuation_token):
                raise FabricError("Pagination repeated a continuing page", error_code="PAGINATION_CYCLE")
            seen_pages.add(signature)
            before = len(items)
            for normalized in page_items:
                item_id = normalized["id"]
                if item_id not in items:
                    items[item_id] = normalized
                elif "properties" not in items[item_id] and "properties" in normalized:
                    items[item_id] = normalized
            page_number += 1
            self._event("DISCOVERY_PAGE", flush=True, collection=collection,
                        page_number=page_number, item_count=len(values),
                        new_item_count=len(items) - before,
                        continuation_uri=continuation_uri, continuation_token=continuation_token)
            if continuation_token:
                if continuation_token in seen_tokens:
                    raise FabricError("Pagination repeated a continuation token",
                                      error_code="PAGINATION_CYCLE")
                seen_tokens.add(continuation_token)
                url = self._url(root + "?" + urlencode({
                    "recursive": "true", "continuationToken": continuation_token,
                }))
            elif continuation_uri:
                url = self._joined_url(url, continuation_uri)
            else:
                return list(items.values())


def _read_status(client, context, phase):
    path = ("/workspaces/" + client.config.workspace_id + "/sqlEndpoints/"
            + context["endpoint_id"] + "/metadataSyncMigrationStatus")
    reply = client.call("GET", path, "MIGRATION_STATUS_" + phase, context)
    if reply.status != 200:
        _contract("Migration status requires HTTP 200")
    status = _status_shape(_object(reply, "Migration status"))
    attempt = status.get("lastMigrationAttempt", {})
    client._event("MIGRATION_STATUS", context, sync_version=status["syncArchitectureVersion"],
                  migration_direction=status.get("itemMigrationDirection"),
                  migration_status=attempt.get("itemMigrationStatus"), status_phase=phase,
                  table_failure_count=len(status["tableFailures"]),
                  last_migration_attempt=attempt or None, event_state="ASSESSED")
    if attempt:
        client._event("MIGRATION_ATTEMPT", context, status_phase=phase,
                      migration_status=attempt["itemMigrationStatus"], last_migration_attempt=attempt)
    for failure in status["tableFailures"]:
        client._event("TABLE_MIGRATION_FAILURE", context, status_phase=phase,
                      table_schema=failure["schema"], table_name=failure["tableName"],
                      sync_version=status["syncArchitectureVersion"],
                      error_code=failure["error"]["errorCode"], message=failure["error"]["message"])
    client.audit.flush()
    return status


def _decision(client, context, summary, outcome, message):
    summary.update(outcome=outcome, message=message)
    client._event("ENDPOINT_DECISION", context, flush=True, event_state=outcome,
                  outcome=outcome, message=message, upgrade_attempted=summary["upgrade_attempted"])


def _assess_endpoint(client, context, summary):
    initial = _read_status(client, context, "INITIAL")
    version = initial["syncArchitectureVersion"]
    summary.update(version_before=version, version_after=version,
                   migration_status=initial.get("lastMigrationAttempt", {}).get("itemMigrationStatus"))
    if initial.get("itemMigrationDirection"):
        _decision(client, context, summary, "MIGRATION_IN_PROGRESS", "An active migration cannot be upgraded")
        return
    if version == "V2" and not initial["tableFailures"] and not _partial_attempt(initial):
        _decision(client, context, summary, "ALREADY_V2", "V2 endpoint has no known remaining V1 failures")
        return
    root = "/workspaces/" + client.config.workspace_id + "/sqlEndpoints/" + context["endpoint_id"]
    try:
        preflight = _preflight_shape(client.operation("POST", root + "/preflightMetadataSyncUpgrade",
                                                      "PREFLIGHT", context))
    except FabricError:
        summary["operation_id"] = client.last_operation_id
        raise
    summary["operation_id"] = client.last_operation_id
    evaluated, eligible, blockers = (preflight["evaluatedTableCount"],
                                    preflight["eligibleTableCount"], preflight["blockers"])
    all_ready = evaluated > 0 and eligible == evaluated and not blockers
    summary.update(evaluated_table_count=evaluated, eligible_table_count=eligible,
                   blocker_count=len(blockers), all_tables_eligible=all_ready)
    client._event("PREFLIGHT_RESULT", context, evaluated_table_count=evaluated,
                  eligible_table_count=eligible, blocker_count=len(blockers),
                  all_tables_eligible=all_ready, parsed_result=preflight)
    for blocker in blockers:
        client._event("TABLE_BLOCKER", context, table_schema=blocker["schema"],
                      table_name=blocker["tableName"], corrective_action=blocker["correctiveAction"],
                      message=blocker["correctiveAction"])
    client.audit.flush()
    if evaluated == 0:
        _decision(client, context, summary, "NO_TABLES", "Preflight evaluated no tables; upgrade is not permitted")
        return
    config = client.config
    if not config.force_upgrade and not config.upgrade_if_all_eligible:
        _decision(client, context, summary, "VALIDATION_READY" if all_ready else "VALIDATION_BLOCKED",
                  "Validation only; both upgrade switches are disabled")
        return
    if not config.force_upgrade and not all_ready:
        _decision(client, context, summary, "UPGRADE_SKIPPED_BLOCKERS",
                  "Safe upgrade requires every evaluated table eligible and no blockers")
        return
    current = _read_status(client, context, "RECHECK")
    summary["version_after"] = current["syncArchitectureVersion"]
    summary["migration_status"] = current.get("lastMigrationAttempt", {}).get("itemMigrationStatus")
    if current.get("itemMigrationDirection"):
        _decision(client, context, summary, "MIGRATION_IN_PROGRESS", "Migration became active after preflight")
        return
    if current["syncArchitectureVersion"] != version:
        _decision(client, context, summary, "STATE_CHANGED_RERUN",
                  "Architecture version changed after preflight; rerun assessment")
        return
    client._event("UPGRADE_DECISION", context, flush=True,
                  event_state="FORCE_UPGRADE" if config.force_upgrade else "SAFE_UPGRADE",
                  upgrade_attempted=False, upgrade_planned=True,
                  force_upgrade=config.force_upgrade, upgrade_if_all_eligible=config.upgrade_if_all_eligible,
                  evaluated_table_count=evaluated, eligible_table_count=eligible,
                  blocker_count=len(blockers), all_tables_eligible=all_ready,
                  message="Client eligibility gate approved; preflight is point-in-time, not a guarantee")
    summary.update(upgrade_attempted=True, migration_status=None, version_after=None)
    result, upgrade_error = None, None
    try:
        result = _upgrade_shape(client.operation("POST", root + "/upgradeMetadataSync", "UPGRADE", context))
        summary["migration_status"] = result["itemMigrationStatus"]
        for table in result["tableStatuses"]:
            error = table.get("error") or {}
            client._event("TABLE_UPGRADE_STATUS", context, table_schema=table["schema"],
                          table_name=table["tableName"], table_sync_version=table["syncArchitectureVersion"],
                          last_successful_sync_utc=table.get("lastSuccessfulSyncDateTime"),
                          sync_version=result["syncArchitectureVersion"],
                          migration_status=result["itemMigrationStatus"], error_code=error.get("errorCode"),
                          message=error.get("message"), operation_id=client.last_operation_id)
        client._event("UPGRADE_RESULT", context, flush=True, parsed_result=result,
                      sync_version=result["syncArchitectureVersion"], migration_status=result["itemMigrationStatus"],
                      operation_id=client.last_operation_id, upgrade_attempted=True)
    except FabricError as exc:
        upgrade_error = exc
        client._event("UPGRADE_ERROR", context, flush=True, event_state="FAILED",
                      upgrade_attempted=True, error_code=exc.error_code, message=str(exc),
                      operation_id=client.last_operation_id)
    summary["operation_id"] = client.last_operation_id
    # Deliberately not finally: audit failures/programming bugs must abort immediately.
    post, post_error = None, None
    try:
        post = _read_status(client, context, "POST_UPGRADE")
        summary["version_after"] = post["syncArchitectureVersion"]
        if summary["migration_status"] is None:
            summary["migration_status"] = post.get("lastMigrationAttempt", {}).get("itemMigrationStatus")
    except FabricError as exc:
        post_error = exc
        client._event("POST_STATUS_ERROR", context, flush=True, event_state="FAILED",
                      upgrade_attempted=True, error_code=exc.error_code, message=str(exc))
    if upgrade_error:
        outcome, message = "UPGRADE_FAILED", "Upgrade outcome could not be verified: " + str(upgrade_error)
    else:
        migration_status = result["itemMigrationStatus"]
        tables = result["tableStatuses"]
        remaining = any(table["syncArchitectureVersion"] != "V2" or table.get("error") for table in tables)
        upgraded = any(table["syncArchitectureVersion"] == "V2" and not table.get("error") for table in tables)
        if migration_status == "Failed":
            outcome, message = "UPGRADE_FAILED", "Upgrade result reports Failed"
        elif migration_status == "PartiallySucceeded" or remaining or result["syncArchitectureVersion"] != "V2":
            outcome = "UPGRADE_PARTIAL" if migration_status == "PartiallySucceeded" or upgraded else "UPGRADE_FAILED"
            message = "Upgrade was not fully successful; remaining V1/error tables do not sync automatically"
        else:
            outcome, message = "UPGRADED", "Result and all reported tables indicate successful V2 upgrade"
        if outcome == "UPGRADED":
            if post_error or post is None or post.get("itemMigrationDirection") or post["syncArchitectureVersion"] != "V2":
                outcome, message = "UPGRADE_FAILED", "Reported success could not be verified by post-upgrade status"
            elif post["tableFailures"] or _partial_attempt(post):
                outcome, message = "UPGRADE_PARTIAL", "Post-upgrade status still reports failures or a partial/failed attempt"
    if post_error:
        message += "; post-upgrade status read also failed: " + str(post_error)
    _decision(client, context, summary, outcome, message)


def run_workspace(client, audit):
    """Discover completely before mutating; isolate FabricError, never audit failures."""
    if audit is not client.audit:
        raise ValueError("run_workspace must use the client's audit sink")
    config = client.config
    client._event("RUN_START", flush=True, event_state="STARTED",
                  upgrade_if_all_eligible=config.upgrade_if_all_eligible,
                  force_upgrade=config.force_upgrade, api_base_url=config.api_base_url)
    try:
        lakehouses = client.list_all("lakehouses")
        listed_endpoints = client.list_all("sqlEndpoints")
        endpoints = {item["id"]: item for item in listed_endpoints}
        associations = {}
        for lakehouse in lakehouses:
            lh_context = {"lakehouse_id": lakehouse["id"], "lakehouse_name": lakehouse.get("displayName")}
            client._event("LAKEHOUSE_DISCOVERED", lh_context)
            if "properties" not in lakehouse:
                reply = client.call("GET", "/workspaces/" + config.workspace_id + "/lakehouses/" + lakehouse["id"],
                                    "LAKEHOUSE_DETAIL", lh_context)
                if reply.status != 200:
                    _contract("Lakehouse detail requires HTTP 200")
                detail = _object(reply, "Lakehouse detail")
                if _uuid(detail.get("id"), "Lakehouse detail ID") != lakehouse["id"]:
                    _contract("Lakehouse detail ID changed")
                if "workspaceId" in detail and _uuid(detail["workspaceId"], "Lakehouse workspace ID") != config.workspace_id:
                    _contract("Lakehouse detail belongs to another workspace")
                lakehouse = detail
            properties = lakehouse.get("properties")
            if not isinstance(properties, dict):
                _contract("Lakehouse properties are missing or malformed")
            sql_properties = properties.get("sqlEndpointProperties")
            if sql_properties is None or (isinstance(sql_properties, dict)
                                          and sql_properties.get("id") in (None, "")):
                client._event("LAKEHOUSE_SKIPPED", lh_context, flush=True,
                              event_state="ENDPOINT_MISSING", outcome="ENDPOINT_MISSING",
                              message="Lakehouse has no provisioned SQL endpoint ID")
                continue
            if not isinstance(sql_properties, dict):
                _contract("sqlEndpointProperties must be an object")
            endpoint_id = _uuid(sql_properties["id"], "Lakehouse SQL endpoint ID")
            provisioning = sql_properties.get("provisioningStatus")
            if provisioning is not None and not isinstance(provisioning, str):
                _contract("SQL endpoint provisioningStatus must be a string")
            context = dict(lh_context, endpoint_id=endpoint_id)
            ready = provisioning == "Success"
            associations.setdefault(endpoint_id, []).append((context, ready, provisioning))
            endpoints.setdefault(endpoint_id, {"id": endpoint_id})
            client._event("ENDPOINT_ASSOCIATION", context, provisioning_status=provisioning, endpoint_ready=ready)
            if not ready:
                client._event("LAKEHOUSE_SKIPPED", context, flush=True,
                              event_state="ENDPOINT_NOT_READY", outcome="ENDPOINT_NOT_READY",
                              provisioning_status=provisioning,
                              message="Lakehouse SQL endpoint provisioning has not reported Success")
        audit.flush()
    except FabricError as exc:
        client._event("DISCOVERY_ERROR", flush=True, event_state="FAILED",
                      error_code=exc.error_code, message=str(exc))
        client._event("RUN_FAILED", flush=True, event_state="FAILED", all_endpoints_processed=False,
                      processed_endpoint_count=0, error_code=exc.error_code,
                      message="Discovery was incomplete; no endpoint upgrades were started")
        raise
    summaries = []
    for endpoint_id, endpoint in endpoints.items():
        links = associations.get(endpoint_id, [])
        chosen = next((link for link in links if link[1]), links[0] if links else None)
        context = dict(chosen[0] if chosen else {}, endpoint_id=endpoint_id,
                       endpoint_name=endpoint.get("displayName"))
        context.setdefault("lakehouse_id", None)
        context.setdefault("lakehouse_name", None)
        summary = dict.fromkeys(_SUMMARY_FIELDS)
        summary.update(context, all_tables_eligible=False, upgrade_attempted=False)
        client._event("ENDPOINT_DISCOVERED", context)
        client._event("ENDPOINT_START", context, flush=True, event_state="STARTED")
        try:
            if chosen is not None and not chosen[1]:
                _decision(client, context, summary, "ENDPOINT_NOT_READY",
                          "Associated lakehouse SQL endpoint is not ready")
            else:
                _assess_endpoint(client, context, summary)
        except FabricError as exc:
            summary.update(outcome="ERROR", message=str(exc))
            client._event("ENDPOINT_ERROR", context, flush=True, event_state="ERROR",
                          error_code=exc.error_code, message=str(exc),
                          upgrade_attempted=summary["upgrade_attempted"], operation_id=summary["operation_id"])
        client._event("ENDPOINT_SUMMARY", context, flush=True,
                      event_state=summary["outcome"], **summary)
        summaries.append(summary)
    counts = {}
    for summary in summaries:
        counts[summary["outcome"]] = counts.get(summary["outcome"], 0) + 1
    has_errors = any(key in counts for key in ("ERROR", "UPGRADE_FAILED", "UPGRADE_PARTIAL"))
    client._event("RUN_COMPLETE", flush=True,
                  event_state="COMPLETE_WITH_ERRORS" if has_errors else "COMPLETE",
                  all_endpoints_processed=True, lakehouse_count=len(lakehouses),
                  listed_endpoint_count=len(listed_endpoints), endpoint_count=len(endpoints),
                  processed_endpoint_count=len(summaries), outcome_counts=counts)
    return summaries


## Runtime, permissions, and durable audit storage

Run with **PySpark (Python)** and keep the linked, schema-enabled `lakehouse`
attached as the default lakehouse. The notebook refuses to write to a different
default lakehouse. The audit table is initialized before the first Fabric API
call; upgrade intent is durably logged before an upgrade request is sent.
An audit write failure stops the run, rather than continuing without a record.

Use an identity with workspace enumeration/read access, write permission on each
SQL analytics endpoint for preflight/upgrade, and write access to the audit
lakehouse. The REST scopes are `Workspace.Read.All`/`Workspace.ReadWrite.All`
for enumeration and `Item.ReadWrite.All` or `SQLEndpoint.ReadWrite.All` for
assessment/upgrade. The migration APIs are currently Preview.

Authentication uses `notebookutils.credentials.getToken("pbi")` at runtime,
refreshed for each request attempt. No credentials are stored in the notebook
or audit table. Service-principal notebook tokens have limited scopes; unattended
execution may require replacing the token provider with an approved MSAL/managed
identity implementation that has the required SQL endpoint scopes.

HTTP 202 operations are polled to a terminal state and their result is fetched.
Throttling respects `Retry-After`. Ambiguous upgrade timeouts/5xx responses are
**not** automatically replayed. An attempted upgrade is followed by another
state query, even on an API/operation error. Operation IDs remain in the audit
table so a timed-out server-side operation can be investigated before rerunning.


### API compatibility (notebook 1.0.1)

Migration attempts accept both the documented long field names and MSIT's `sourceVersion`, `targetVersion`, `status`, and `completedAt` fields. Known migration outcomes are normalized case-insensitively for consistent review. Conflicting aliases and unknown values still stop assessment or upgrade; full original response bodies remain in the audit table.


In [ ]:
import json
import re
import uuid
from datetime import datetime, timezone

import notebookutils
import requests
from py4j.protocol import Py4JError
from pyspark.errors import PySparkException
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, StringType, StructField, StructType,
    TimestampType,
)


class AuditWriteError(RuntimeError):
    pass


def quote_identifier(value):
    if not isinstance(value, str) or not value:
        raise ValueError("Audit identifiers must be nonempty strings.")
    return "`" + value.replace("`", "``") + "`"


if type(FAIL_ON_ERRORS) is not bool:
    raise ValueError("FAIL_ON_ERRORS must be a Python boolean.")
for identifier in (AUDIT_SCHEMA_NAME, AUDIT_TABLE_NAME):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError("Audit schema/table names must contain letters, numbers, underscores.")

config = Config(
    workspace_id=WORKSPACE_ID,
    api_base_url=FABRIC_API_BASE_URL,
    upgrade_if_all_eligible=UPGRADE_IF_ALL_ELIGIBLE,
    force_upgrade=FORCE_UPGRADE,
    http_timeout_seconds=HTTP_TIMEOUT_SECONDS,
    operation_timeout_seconds=OPERATION_TIMEOUT_SECONDS,
    poll_interval_seconds=POLL_INTERVAL_SECONDS,
    max_retries=MAX_RETRIES,
)

context = notebookutils.runtime.context
if (
    str(context.get("defaultLakehouseId", "")).lower() != AUDIT_LAKEHOUSE_ID.lower()
    or str(context.get("defaultLakehouseWorkspaceId", "")).lower()
    != AUDIT_LAKEHOUSE_WORKSPACE_ID.lower()
    or context.get("defaultLakehouseName") != AUDIT_LAKEHOUSE_NAME
):
    raise AuditWriteError(
        "Attach the configured audit lakehouse as the default lakehouse and restart "
        "the Spark session before running. No Fabric API calls have been made."
    )

spark.conf.set("spark.sql.session.timeZone", "UTC")
AUDIT_TABLE_FQN = ".".join(
    quote_identifier(part)
    for part in (AUDIT_LAKEHOUSE_NAME, AUDIT_SCHEMA_NAME, AUDIT_TABLE_NAME)
)
RUN_ID = str(uuid.uuid4())
NOTEBOOK_VERSION = "1.0.1"

AUDIT_PROJECTIONS = [
    ("endpoint_id", StringType()),
    ("endpoint_name", StringType()),
    ("lakehouse_id", StringType()),
    ("lakehouse_name", StringType()),
    ("event_state", StringType()),
    ("http_method", StringType()),
    ("request_url", StringType()),
    ("http_status", IntegerType()),
    ("request_id", StringType()),
    ("operation_id", StringType()),
    ("operation_state", StringType()),
    ("sync_version", StringType()),
    ("version_before", StringType()),
    ("version_after", StringType()),
    ("migration_direction", StringType()),
    ("migration_status", StringType()),
    ("table_schema", StringType()),
    ("table_name", StringType()),
    ("table_sync_version", StringType()),
    ("last_successful_sync_utc", StringType()),
    ("evaluated_table_count", LongType()),
    ("eligible_table_count", LongType()),
    ("blocker_count", LongType()),
    ("all_tables_eligible", BooleanType()),
    ("upgrade_attempted", BooleanType()),
    ("outcome", StringType()),
    ("error_code", StringType()),
    ("message", StringType()),
    ("response_body", StringType()),
    ("response_headers_json", StringType()),
]
AUDIT_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("event_id", StringType(), False),
    StructField("event_sequence", LongType(), False),
    StructField("event_time_utc", TimestampType(), False),
    StructField("workspace_id", StringType(), False),
    StructField("audit_lakehouse_id", StringType(), False),
    StructField("notebook_id", StringType(), True),
    StructField("notebook_version", StringType(), False),
    StructField("stage", StringType(), False),
    StructField("upgrade_if_all_eligible", BooleanType(), False),
    StructField("force_upgrade", BooleanType(), False),
] + [
    StructField(name, datatype, True) for name, datatype in AUDIT_PROJECTIONS
] + [StructField("details_json", StringType(), False)])


class DeltaAudit:
    def __init__(self):
        self.pending = []
        self.sequence = 0
        try:
            spark.createDataFrame([], AUDIT_SCHEMA).write.format("delta").mode(
                "ignore"
            ).saveAsTable(AUDIT_TABLE_FQN)
            expected = [(f.name, f.dataType) for f in AUDIT_SCHEMA.fields]
            actual = [(f.name, f.dataType) for f in spark.table(AUDIT_TABLE_FQN).schema.fields]
            if actual != expected:
                raise AuditWriteError(
                    "Existing audit table schema differs from this notebook. Choose a new "
                    "AUDIT_TABLE_NAME or explicitly migrate the existing table; no API calls made."
                )
        except (PySparkException, Py4JError, OSError, ValueError, TypeError) as exc:
            raise AuditWriteError(
                f"Unable to initialize the Delta audit table {AUDIT_TABLE_FQN}: {exc}"
            ) from exc

    def emit(self, stage, **fields):
        self.sequence += 1
        row = {
            "run_id": RUN_ID,
            "event_id": str(uuid.uuid4()),
            "event_sequence": self.sequence,
            "event_time_utc": datetime.now(timezone.utc).replace(tzinfo=None),
            "workspace_id": WORKSPACE_ID,
            "audit_lakehouse_id": AUDIT_LAKEHOUSE_ID,
            "notebook_id": context.get("currentNotebookId"),
            "notebook_version": NOTEBOOK_VERSION,
            "stage": stage,
            "upgrade_if_all_eligible": UPGRADE_IF_ALL_ELIGIBLE,
            "force_upgrade": FORCE_UPGRADE,
            "details_json": json.dumps(fields, ensure_ascii=True, allow_nan=False),
        }
        for name, datatype in AUDIT_PROJECTIONS:
            value = fields.get(name)
            if value is not None and isinstance(datatype, StringType) and not isinstance(value, str):
                value = json.dumps(value, ensure_ascii=True, allow_nan=False)
            row[name] = value
        self.pending.append(row)

    def flush(self):
        if not self.pending:
            return
        count = len(self.pending)
        try:
            spark.createDataFrame(self.pending[:count], AUDIT_SCHEMA).write.format(
                "delta"
            ).mode("append").saveAsTable(AUDIT_TABLE_FQN)
        except (PySparkException, Py4JError, OSError, ValueError, TypeError) as exc:
            raise AuditWriteError(
                f"Audit persistence failed for run {RUN_ID}. The run is stopped; "
                "inspect the last logged operation/state before retrying any upgrade. "
                f"Details: {exc}"
            ) from exc
        del self.pending[:count]


audit = DeltaAudit()
audit.emit(
    "NOTEBOOK_CONTEXT",
    event_state="Initialized",
    message="Audit storage initialized; upgrades require explicit parameter switches.",
    audit_table=AUDIT_TABLE_FQN,
    current_workspace_id=context.get("currentWorkspaceId"),
    is_pipeline=context.get("isForPipeline"),
    is_interactive=context.get("isForInteractive"),
    api_base_url=FABRIC_API_BASE_URL,
)
audit.flush()


def get_fabric_token():
    return notebookutils.credentials.getToken("pbi")


http_session = requests.Session()


def fabric_transport(method, url, headers, timeout_seconds):
    try:
        with http_session.request(
            method, url, headers=headers, timeout=timeout_seconds, allow_redirects=False
        ) as response:
            return HttpReply(
                status=response.status_code,
                headers=dict(response.headers),
                text=response.text,
            )
    except requests.exceptions.RequestException as exc:
        raise TransportError(f"{type(exc).__name__}: {exc}") from exc


client = Client(config, get_fabric_token, fabric_transport, audit)
print(f"Run: {RUN_ID}")
print(f"Audit: {AUDIT_TABLE_FQN}")
print(
    "Mode: "
    + ("FORCED UPGRADE" if FORCE_UPGRADE else
       "UPGRADE IF ALL TABLES ELIGIBLE" if UPGRADE_IF_ALL_ELIGIBLE else "VALIDATION ONLY")
)


## Run workspace assessment / optional upgrades

Every endpoint produces a summary. Operational errors are logged and other endpoints continue; any audit persistence failure stops the entire run.


In [ ]:
try:
    endpoint_summaries = run_workspace(client, audit)
finally:
    try:
        audit.flush()
    finally:
        http_session.close()

print(f"Processed {len(endpoint_summaries)} SQL analytics endpoint(s).")
print(f"Responses and decisions persisted to {AUDIT_TABLE_FQN}, run_id={RUN_ID}.")


## Review this run and historical results

The following cell reads the durable audit table, not an in-memory substitute.
Use `run_id` to compare runs, `endpoint_id` for item history, and `operation_id`
to correlate asynchronous requests. Raw response bodies are not truncated;
use Spark for full JSON when responses are large.

`ENDPOINT_SUMMARY` has one row per endpoint. Rows with `table_name` identify
individual assessment blockers, prior migration failures, and upgrade results.
The `details_json` column retains all additional structured event fields.
The audit table is itself a Delta table in the linked lakehouse and may therefore
also be included in that endpoint's assessment.

With `FAIL_ON_ERRORS=True`, API/contract errors and partial/failed upgrades fail
the notebook **after** results are persisted and displayed. Eligibility blockers
in validation-only mode are expected assessment outcomes, not execution errors.

Example T-SQL from the linked SQL analytics endpoint once the audit table syncs:

```sql
SELECT run_id, event_time_utc, endpoint_name, version_before, version_after,
       evaluated_table_count, eligible_table_count, blocker_count,
       upgrade_attempted, outcome, message
FROM dbo.md_sync_audit
WHERE stage = 'ENDPOINT_SUMMARY'
ORDER BY event_time_utc DESC;

SELECT run_id, endpoint_name, table_schema, table_name,
       table_sync_version, error_code, message, response_body
FROM dbo.md_sync_audit
WHERE table_name IS NOT NULL
ORDER BY event_time_utc DESC;
```


In [ ]:
run_audit = spark.table(AUDIT_TABLE_FQN).where(F.col("run_id") == RUN_ID)

# Endpoint-level summary for this run
display(
    run_audit.where(F.col("stage") == "ENDPOINT_SUMMARY")
    .select(
        "endpoint_name", "endpoint_id", "lakehouse_name",
        "version_before", "version_after", "evaluated_table_count",
        "eligible_table_count", "blocker_count", "all_tables_eligible",
        "upgrade_attempted", "outcome", "message",
    )
    .orderBy("endpoint_name", "endpoint_id")
)

# All table-level events for this run (blockers, previous failures, upgrade results)
display(
    run_audit.where(F.col("table_name").isNotNull())
    .select(
        "event_sequence", "stage", "endpoint_name", "table_schema",
        "table_name", "table_sync_version", "error_code", "message",
    )
    .orderBy("event_sequence")
)

# Full event timeline for this run (for troubleshooting)
display(
    run_audit.select(
        "event_sequence", "event_time_utc", "stage", "event_state",
        "endpoint_name", "http_status", "operation_id", "operation_state",
        "sync_version", "migration_direction", "migration_status", "message",
    ).orderBy("event_sequence")
)

# ---- Additional diagnostics: upgraded tables and failures for this run ----

# Tables successfully upgraded to V2 in this run
upgraded_tables = (
    run_audit
    .where(F.col("stage") == "TABLE_UPGRADE_STATUS")
    .where(F.col("table_sync_version") == "V2")
    .where(F.col("error_code").isNull())
    .select(
        "endpoint_name", "endpoint_id", "table_schema", "table_name",
        "table_sync_version", "migration_status", "operation_id",
    )
    .orderBy("endpoint_name", "table_schema", "table_name")
)
display(upgraded_tables)

# Tables with blockers or errors (preflight blockers, migration failures, upgrade errors)
problem_tables = (
    run_audit
    .where(F.col("table_name").isNotNull())
    .where(
        (F.col("error_code").isNotNull())
        | (F.col("stage").isin("TABLE_BLOCKER", "TABLE_MIGRATION_FAILURE"))
        | (
            (F.col("stage") == "TABLE_UPGRADE_STATUS")
            & ((F.col("table_sync_version") != "V2") | F.col("error_code").isNotNull())
        )
    )
    .select(
        "endpoint_name", "endpoint_id", "table_schema", "table_name",
        "stage", "table_sync_version", "error_code", "message", "operation_id",
    )
    .orderBy("endpoint_name", "table_schema", "table_name", "stage")
)
display(problem_tables)

# Endpoint summaries that had errors or incomplete upgrades (directly from audit table)
error_outcomes = ["ERROR", "UPGRADE_PARTIAL", "UPGRADE_FAILED", "STATE_CHANGED_RERUN"]
unsuccessful_df = (
    run_audit
    .where(F.col("stage") == "ENDPOINT_SUMMARY")
    .where(F.col("outcome").isin(error_outcomes) | F.col("outcome").startswith("ERROR"))
    .select(
        "endpoint_name", "endpoint_id", "lakehouse_name", "version_before",
        "version_after", "evaluated_table_count", "eligible_table_count",
        "blocker_count", "upgrade_attempted", "outcome", "message", "operation_id",
    )
)

count_unsuccessful = unsuccessful_df.count()
if count_unsuccessful:
    print(f"{count_unsuccessful} endpoint(s) had errors or incomplete upgrades in this run.")
    display(unsuccessful_df.orderBy("endpoint_name", "endpoint_id"))

# Fail the notebook (after showing details) if requested
if FAIL_ON_ERRORS and count_unsuccessful:
    raise RuntimeError(
        f"{count_unsuccessful} endpoint(s) had errors or incomplete upgrades. "
        f"Review {AUDIT_TABLE_FQN} for run_id={RUN_ID}; no results were discarded."
    )

if context.get("isForPipeline") or context.get("isForInteractive") is False:
    notebookutils.notebook.exit(json.dumps({
        "run_id": RUN_ID,
        "audit_table": AUDIT_TABLE_FQN,
        "upgrade_if_all_eligible": UPGRADE_IF_ALL_ELIGIBLE,
        "force_upgrade": FORCE_UPGRADE,
        "endpoint_summaries": endpoint_summaries,
    }, ensure_ascii=True))


## Official references and blocker remediation

The notebook uses the documented REST contracts available on 21 November 2026:

- [Upgrade existing SQL analytics endpoints to new metadata sync](https://learn.microsoft.com/en-us/fabric/data-engineering/sql-analytics-endpoint-metadata-sync#upgrade-existing-sql-analytics-endpoints-to-new-metadata-sync)
- [Get metadata sync migration status](https://learn.microsoft.com/en-us/rest/api/fabric/sqlendpoint/items/get-sql-endpoint-metadata-sync-migration-status)
- [Preflight metadata sync upgrade](https://learn.microsoft.com/en-us/rest/api/fabric/sqlendpoint/items/preflight-sql-endpoint-metadata-sync-upgrade)
- [Upgrade metadata sync](https://learn.microsoft.com/en-us/rest/api/fabric/sqlendpoint/items/upgrade-sql-endpoint-metadata-sync)
- [List SQL analytics endpoints](https://learn.microsoft.com/en-us/rest/api/fabric/sqlendpoint/items/list-sql-endpoints)
- [List lakehouses](https://learn.microsoft.com/en-us/rest/api/fabric/lakehouse/items/list-lakehouses)
- [Fabric long-running operations](https://learn.microsoft.com/en-us/rest/api/fabric/articles/long-running-operation)
- [NotebookUtils credentials and scope limitations](https://learn.microsoft.com/en-us/fabric/data-engineering/notebookutils/notebookutils-credentials)

Resolve the API's per-table `correctiveAction` before using the safe upgrade switch.
Known blockers include deprecated multi-part Delta checkpoints, more than 1,000
commits since the last checkpoint, and column mapping by ID. Delta checkpoint
policies `classic` and `v2` are separate from metadata sync architecture V1/V2.
Do not assume changing `delta.columnMapping.mode` directly is supported; follow
the managing runtime's documented migration procedure and control concurrent
writers. This notebook intentionally performs no such underlying data changes.
